##  1. Configuration et Imports

In [1]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

# Tentative d'import de geopandas pour les shapefiles
try:
    import geopandas as gpd
    from shapely.geometry import Point, Polygon
    GEO_SUPPORT = True
    print(" geopandas disponible - Support géospatial activé")
except ImportError:
    GEO_SUPPORT = False
    print(" geopandas non disponible - Installation requise: pip install geopandas")

# Configuration des chemins
BASE_PATH = '/home/henintsoa/CFIM'
CSV_PATH = os.path.join(BASE_PATH, 'csv')
SHAPEFILE_PATH = os.path.join(BASE_PATH, 'Mdg_COD_Boundry_Shapes_BNGRC_OCHA_Dec17')
OUTPUT_PATH = os.path.join(BASE_PATH, 'final/data')

print(f"\n Chemins configurés:")
print(f"   • Base: {BASE_PATH}")
print(f"   • Shapefiles: {SHAPEFILE_PATH}")
print(f"   • Output: {OUTPUT_PATH}")

 geopandas disponible - Support géospatial activé

 Chemins configurés:
   • Base: /home/henintsoa/CFIM
   • Shapefiles: /home/henintsoa/CFIM/Mdg_COD_Boundry_Shapes_BNGRC_OCHA_Dec17
   • Output: /home/henintsoa/CFIM/final/data


##  2. Identification des Zones Côtières

Les bulletins météo marins de Madagascar divisent la côte en plusieurs **zones** définies par des points de repère géographiques.

#### 2.1 Definitons des zones cotieres

Les zones côtières de Madagascar telles qu'utilisées dans les bulletins météo avec leurs coordonnées approximatives des points de repère

In [2]:
POINTS_REPERE = {
    "CAP D'AMBRE": {"lat": -11.95, "lon": 49.27, "description": "Pointe nord de Madagascar"},
    "ANTALAHA": {"lat": -14.90, "lon": 50.28, "description": "Côte nord-est"},
    "TOAMASINA": {"lat": -18.15, "lon": 49.42, "description": "Principal port est"},
    "MAHANORO": {"lat": -19.90, "lon": 48.80, "description": "Côte est centrale"},
    "MANANJARY": {"lat": -21.23, "lon": 48.35, "description": "Côte sud-est"},
    "CAP SAINTE MARIE": {"lat": -25.58, "lon": 45.17, "description": "Pointe sud de Madagascar"},
    "TAOLAGNARO": {"lat": -25.03, "lon": 47.00, "description": "Fort-Dauphin, sud-est"},
    "CAP EST": {"lat": -15.27, "lon": 50.48, "description": "Pointe est (Masoala)"},
    "BESALAMPY": {"lat": -16.75, "lon": 44.48, "description": "Côte ouest-nord"},
    "MAJUNGA": {"lat": -15.72, "lon": 46.32, "description": "Mahajanga, côte nord-ouest"},
    "MOROMBE": {"lat": -21.75, "lon": 43.37, "description": "Côte sud-ouest"},
    "TULEAR": {"lat": -23.35, "lon": 43.67, "description": "Toliara, côte sud-ouest"}
}

print(" POINTS DE REPÈRE CÔTIERS DE MADAGASCAR")
print("=" * 70)
df_points = pd.DataFrame(POINTS_REPERE).T
df_points = df_points.reset_index().rename(columns={'index': 'Point'})
display(df_points)

 POINTS DE REPÈRE CÔTIERS DE MADAGASCAR


,Point,lat,lon,description
0,CAP D'AMBRE,-11.95,49.27,Pointe nord de Madagascar
1,ANTALAHA,-14.9,50.28,Côte nord-est
2,TOAMASINA,-18.15,49.42,Principal port est
3,MAHANORO,-19.9,48.8,Côte est centrale
4,MANANJARY,-21.23,48.35,Côte sud-est
5,CAP SAINTE MARIE,-25.58,45.17,Pointe sud de Madagascar
6,TAOLAGNARO,-25.03,47.0,"Fort-Dauphin, sud-est"
7,CAP EST,-15.27,50.48,Pointe est (Masoala)
8,BESALAMPY,-16.75,44.48,Côte ouest-nord
9,MAJUNGA,-15.72,46.32,"Mahajanga, côte nord-ouest"


#### 2.2 Definition des zones cotieres et leurs segments

Chaque zone côtière est définie par un segment de la côte entre deux points de repère

In [3]:
ZONES_COTIERES = {
    # ===== CÔTE EST =====
    "CAP D'AMBRE A TOAMASINA": {
        "cote": "EST",
        "point_debut": "CAP D'AMBRE",
        "point_fin": "TOAMASINA",
        "lat_min": -18.5,
        "lat_max": -11.5,
        "lon_min": 49.0,
        "lon_max": 50.5,
        "description": "Côte nord-est, de la pointe nord jusqu'à Toamasina"
    },
    "CAP D'AMBRE A MAHANORO": {
        "cote": "EST",
        "point_debut": "CAP D'AMBRE",
        "point_fin": "MAHANORO",
        "lat_min": -20.0,
        "lat_max": -11.5,
        "lon_min": 48.5,
        "lon_max": 50.5,
        "description": "Toute la côte est nord"
    },
    "CAP D'AMBRE A ANTALAHA": {
        "cote": "EST",
        "point_debut": "CAP D'AMBRE",
        "point_fin": "ANTALAHA",
        "lat_min": -15.0,
        "lat_max": -11.5,
        "lon_min": 49.0,
        "lon_max": 50.5,
        "description": "Extrême nord-est"
    },
    "TOAMASINA AU CAP SAINTE MARIE": {
        "cote": "EST-SUD",
        "point_debut": "TOAMASINA",
        "point_fin": "CAP SAINTE MARIE",
        "lat_min": -25.6,
        "lat_max": -18.0,
        "lon_min": 45.0,
        "lon_max": 49.5,
        "description": "Côte est-sud, de Toamasina à la pointe sud"
    },
    "TOAMASINA A TAOLAGNARO": {
        "cote": "EST-SUD",
        "point_debut": "TOAMASINA",
        "point_fin": "TAOLAGNARO",
        "lat_min": -25.1,
        "lat_max": -18.0,
        "lon_min": 47.0,
        "lon_max": 49.5,
        "description": "Côte est, de Toamasina à Fort-Dauphin"
    },
    "MAHANORO AU CAP SAINTE MARIE": {
        "cote": "EST-SUD",
        "point_debut": "MAHANORO",
        "point_fin": "CAP SAINTE MARIE",
        "lat_min": -25.6,
        "lat_max": -19.8,
        "lon_min": 45.0,
        "lon_max": 48.8,
        "description": "Côte sud-est"
    },
    
    # ===== CÔTE OUEST =====
    "CAP D'AMBRE A BESALAMPY": {
        "cote": "OUEST-NORD",
        "point_debut": "CAP D'AMBRE",
        "point_fin": "BESALAMPY",
        "lat_min": -17.0,
        "lat_max": -11.5,
        "lon_min": 44.0,
        "lon_max": 49.5,
        "description": "Côte nord-ouest"
    },
    "BESALAMPY A MOROMBE": {
        "cote": "OUEST",
        "point_debut": "BESALAMPY",
        "point_fin": "MOROMBE",
        "lat_min": -22.0,
        "lat_max": -16.5,
        "lon_min": 43.0,
        "lon_max": 45.0,
        "description": "Côte ouest centrale"
    },
    "MOROMBE AU CAP SAINTE MARIE": {
        "cote": "OUEST-SUD",
        "point_debut": "MOROMBE",
        "point_fin": "CAP SAINTE MARIE",
        "lat_min": -25.6,
        "lat_max": -21.5,
        "lon_min": 43.0,
        "lon_max": 45.5,
        "description": "Côte sud-ouest"
    },
    "MOROMBE A TAOLAGNARO": {
        "cote": "SUD",
        "point_debut": "MOROMBE",
        "point_fin": "TAOLAGNARO",
        "lat_min": -25.5,
        "lat_max": -21.5,
        "lon_min": 43.0,
        "lon_max": 47.5,
        "description": "Côte sud complète"
    },
    
    # ===== ZONES SPÉCIALES =====
    "CAP D'AMBRE A CAP EST": {
        "cote": "NORD-EST",
        "point_debut": "CAP D'AMBRE",
        "point_fin": "CAP EST",
        "lat_min": -15.5,
        "lat_max": -11.5,
        "lon_min": 49.0,
        "lon_max": 50.5,
        "description": "Extrême nord-est incluant Masoala"
    },
    "CAP EST A TAOLAGNARO": {
        "cote": "EST",
        "point_debut": "CAP EST",
        "point_fin": "TAOLAGNARO",
        "lat_min": -25.1,
        "lat_max": -15.0,
        "lon_min": 47.0,
        "lon_max": 50.5,
        "description": "Toute la côte est"
    }
}

print(" ZONES CÔTIÈRES DÉFINIES")
print("=" * 70)
for zone, info in ZONES_COTIERES.items():
    print(f"\n {zone}")
    print(f"   Côte: {info['cote']}")
    print(f"   {info['description']}")
    print(f"   Lat: [{info['lat_max']}° à {info['lat_min']}°]")
    print(f"   Lon: [{info['lon_min']}° à {info['lon_max']}°]")

 ZONES CÔTIÈRES DÉFINIES

 CAP D'AMBRE A TOAMASINA
   Côte: EST
   Côte nord-est, de la pointe nord jusqu'à Toamasina
   Lat: [-11.5° à -18.5°]
   Lon: [49.0° à 50.5°]

 CAP D'AMBRE A MAHANORO
   Côte: EST
   Toute la côte est nord
   Lat: [-11.5° à -20.0°]
   Lon: [48.5° à 50.5°]

 CAP D'AMBRE A ANTALAHA
   Côte: EST
   Extrême nord-est
   Lat: [-11.5° à -15.0°]
   Lon: [49.0° à 50.5°]

 TOAMASINA AU CAP SAINTE MARIE
   Côte: EST-SUD
   Côte est-sud, de Toamasina à la pointe sud
   Lat: [-18.0° à -25.6°]
   Lon: [45.0° à 49.5°]

 TOAMASINA A TAOLAGNARO
   Côte: EST-SUD
   Côte est, de Toamasina à Fort-Dauphin
   Lat: [-18.0° à -25.1°]
   Lon: [47.0° à 49.5°]

 MAHANORO AU CAP SAINTE MARIE
   Côte: EST-SUD
   Côte sud-est
   Lat: [-19.8° à -25.6°]
   Lon: [45.0° à 48.8°]

 CAP D'AMBRE A BESALAMPY
   Côte: OUEST-NORD
   Côte nord-ouest
   Lat: [-11.5° à -17.0°]
   Lon: [44.0° à 49.5°]

 BESALAMPY A MOROMBE
   Côte: OUEST
   Côte ouest centrale
   Lat: [-16.5° à -22.0°]
   Lon: [43.0° à 

##  3. Chargement des Régions Administratives

Nous allons charger les shapefiles des **22 régions de Madagascar** pour pouvoir faire le mapping.

In [4]:
if GEO_SUPPORT:
    # Charger le shapefile des régions
    shapefile_regions = os.path.join(SHAPEFILE_PATH, 'mdg_polbnda_adm1_Regions_BNGRC_OCHA.shp')
    
    if os.path.exists(shapefile_regions):
        gdf_regions = gpd.read_file(shapefile_regions)
        
        print(" RÉGIONS DE MADAGASCAR CHARGÉES")
        print("=" * 70)
        print(f"Nombre de régions: {len(gdf_regions)}")
        print(f"Colonnes: {list(gdf_regions.columns)}")
        
        # Afficher les régions
        print("\n Liste des régions:")
        for i, region in enumerate(gdf_regions['REGION_NAM'].values, 1):
            print(f"   {i:2d}. {region}")
    else:
        print(f" Fichier shapefile non trouvé: {shapefile_regions}")
        gdf_regions = None
else:
    print(" geopandas non disponible - Utilisation du mapping manuel")
    gdf_regions = None

 RÉGIONS DE MADAGASCAR CHARGÉES
Nombre de régions: 22
Colonnes: ['REG_PCODE', 'R_CODE', 'REGION_NAM', 'BNGRC_R_CO', 'BNGRC_REG_', 'REG_FKT_SH', 'PROV_CODE', 'OLD_PROVIN', 'Source', 'Notes', 'Shape_Leng', 'Shape_Area', 'geometry']

 Liste des régions:
    1. Analamanga
    2. Vakinankaratra
    3. Itasy
    4. Bongolava
    5. Haute Matsiatra
    6. Amoron I Mania
    7. Vatovavy Fitovinany
    8. Ihorombe
    9. Atsimo Atsinanana
   10. Atsinanana
   11. Analanjirofo
   12. Alaotra Mangoro
   13. Boeny
   14. Sofia
   15. Betsiboka
   16. Melaky
   17. Atsimo Andrefana
   18. Androy
   19. Anosy
   20. Menabe
   21. Diana
   22. Sava


##  4. Mapping Zones Côtières ↔ Régions

Nous allons maintenant créer la correspondance entre les zones côtières et les régions administratives. Cette correspondance est basée sur la **géographie côtière** de Madagascar.

### Logique du mapping :
- Une zone côtière peut couvrir **plusieurs régions**
- Une région peut être couverte par **plusieurs zones** (selon les bulletins)
- Les régions **intérieures** (sans littoral) ne sont pas mappées

#### 4.1 DÉFINITION DU MAPPING ZONES ↔ RÉGIONS

Ce mapping est basé sur l'analyse géographique de Madagascar, les régions côtières sont mappées aux zones des bulletins météo

In [5]:
MAPPING_ZONES_REGIONS = {
    # ===== CÔTE NORD-EST =====
    "CAP D'AMBRE A TOAMASINA": {
        "regions": ["DIANA", "SAVA", "ANALANJIROFO", "ATSINANANA"],
        "districts_cles": ["Antsiranana I", "Antsiranana II", "Sambava", "Antalaha", 
                          "Maroantsetra", "Mananara Nord", "Toamasina I", "Toamasina II"],
        "description": "Côte nord-est de Madagascar"
    },
    "CAP D'AMBRE A MAHANORO": {
        "regions": ["DIANA", "SAVA", "ANALANJIROFO", "ATSINANANA"],
        "districts_cles": ["Antsiranana I", "Antsiranana II", "Sambava", "Antalaha",
                          "Maroantsetra", "Toamasina I", "Toamasina II", "Brickaville"],
        "description": "Côte nord-est étendue jusqu'à Mahanoro"
    },
    "CAP D'AMBRE A ANTALAHA": {
        "regions": ["DIANA", "SAVA"],
        "districts_cles": ["Antsiranana I", "Antsiranana II", "Sambava", "Antalaha", "Vohemar"],
        "description": "Extrême nord-est"
    },
    "CAP D'AMBRE A CAP EST": {
        "regions": ["DIANA", "SAVA"],
        "districts_cles": ["Antsiranana I", "Antsiranana II", "Sambava", "Antalaha"],
        "description": "Pointe nord jusqu'au Cap Est (Masoala)"
    },
    
    # ===== CÔTE EST-SUD =====
    "TOAMASINA AU CAP SAINTE MARIE": {
        "regions": ["ATSINANANA", "VATOVAVY FITOVINANY", "ATSIMO ATSINANANA", "ANOSY", "ANDROY"],
        "districts_cles": ["Toamasina I", "Toamasina II", "Brickaville", "Mahanoro",
                          "Mananjary", "Manakara", "Farafangana", "Vangaindrano", 
                          "Taolagnaro", "Amboasary"],
        "description": "Toute la côte est-sud"
    },
    "TOAMASINA A TAOLAGNARO": {
        "regions": ["ATSINANANA", "VATOVAVY FITOVINANY", "ATSIMO ATSINANANA", "ANOSY"],
        "districts_cles": ["Toamasina I", "Toamasina II", "Mahanoro", "Mananjary",
                          "Manakara", "Farafangana", "Vangaindrano", "Taolagnaro"],
        "description": "Côte est de Toamasina à Fort-Dauphin"
    },
    "MAHANORO AU CAP SAINTE MARIE": {
        "regions": ["ATSINANANA", "VATOVAVY FITOVINANY", "ATSIMO ATSINANANA", "ANOSY", "ANDROY"],
        "districts_cles": ["Mahanoro", "Mananjary", "Manakara", "Farafangana",
                          "Vangaindrano", "Taolagnaro", "Amboasary"],
        "description": "Côte sud-est de Mahanoro au Cap Sainte Marie"
    },
    "CAP EST A TAOLAGNARO": {
        "regions": ["ANALANJIROFO", "ATSINANANA", "VATOVAVY FITOVINANY", "ATSIMO ATSINANANA", "ANOSY"],
        "districts_cles": ["Maroantsetra", "Mananara Nord", "Toamasina I", "Toamasina II",
                          "Mahanoro", "Mananjary", "Manakara", "Farafangana", "Taolagnaro"],
        "description": "Toute la côte est"
    },
    
    # ===== CÔTE OUEST-NORD =====
    "CAP D'AMBRE A BESALAMPY": {
        "regions": ["DIANA", "SOFIA", "BOENY", "MELAKY"],
        "districts_cles": ["Antsiranana I", "Antsiranana II", "Ambanja", "Nosy Be",
                          "Analalava", "Mahajanga I", "Mahajanga II", "Mitsinjo", "Besalampy"],
        "description": "Côte nord-ouest"
    },
    
    # ===== CÔTE OUEST =====
    "BESALAMPY A MOROMBE": {
        "regions": ["MELAKY", "MENABE", "ATSIMO ANDREFANA"],
        "districts_cles": ["Besalampy", "Maintirano", "Morondava", "Belo sur Tsiribihina",
                          "Morombe"],
        "description": "Côte ouest centrale"
    },
    
    # ===== CÔTE SUD-OUEST =====
    "MOROMBE AU CAP SAINTE MARIE": {
        "regions": ["ATSIMO ANDREFANA", "ANDROY"],
        "districts_cles": ["Morombe", "Tulear I", "Tulear II", "Ankazoabo", "Ampanihy", 
                          "Beloha", "Tsihombe"],
        "description": "Côte sud-ouest"
    },
    "MOROMBE A TAOLAGNARO": {
        "regions": ["ATSIMO ANDREFANA", "ANDROY", "ANOSY"],
        "districts_cles": ["Morombe", "Tulear I", "Tulear II", "Ampanihy", "Beloha",
                          "Tsihombe", "Amboasary", "Taolagnaro"],
        "description": "Côte sud complète de Morombe à Fort-Dauphin"
    }
}

print("🔗 MAPPING ZONES CÔTIÈRES ↔ RÉGIONS")
print("=" * 70)

for zone, info in MAPPING_ZONES_REGIONS.items():
    print(f"\n {zone}")
    print(f"    Régions: {', '.join(info['regions'])}")

🔗 MAPPING ZONES CÔTIÈRES ↔ RÉGIONS

 CAP D'AMBRE A TOAMASINA
    Régions: DIANA, SAVA, ANALANJIROFO, ATSINANANA

 CAP D'AMBRE A MAHANORO
    Régions: DIANA, SAVA, ANALANJIROFO, ATSINANANA

 CAP D'AMBRE A ANTALAHA
    Régions: DIANA, SAVA

 CAP D'AMBRE A CAP EST
    Régions: DIANA, SAVA

 TOAMASINA AU CAP SAINTE MARIE
    Régions: ATSINANANA, VATOVAVY FITOVINANY, ATSIMO ATSINANANA, ANOSY, ANDROY

 TOAMASINA A TAOLAGNARO
    Régions: ATSINANANA, VATOVAVY FITOVINANY, ATSIMO ATSINANANA, ANOSY

 MAHANORO AU CAP SAINTE MARIE
    Régions: ATSINANANA, VATOVAVY FITOVINANY, ATSIMO ATSINANANA, ANOSY, ANDROY

 CAP EST A TAOLAGNARO
    Régions: ANALANJIROFO, ATSINANANA, VATOVAVY FITOVINANY, ATSIMO ATSINANANA, ANOSY

 CAP D'AMBRE A BESALAMPY
    Régions: DIANA, SOFIA, BOENY, MELAKY

 BESALAMPY A MOROMBE
    Régions: MELAKY, MENABE, ATSIMO ANDREFANA

 MOROMBE AU CAP SAINTE MARIE
    Régions: ATSIMO ANDREFANA, ANDROY

 MOROMBE A TAOLAGNARO
    Régions: ATSIMO ANDREFANA, ANDROY, ANOSY


##  5. Création du DataFrame de Correspondance

In [6]:
correspondances = []

for zone, info in MAPPING_ZONES_REGIONS.items():
    zone_info = ZONES_COTIERES.get(zone, {})
    
    for region in info['regions']:
        correspondances.append({
            'zone_cotiere': zone,
            'region': region,
            'cote': zone_info.get('cote', ''),
            'lat_min': zone_info.get('lat_min', None),
            'lat_max': zone_info.get('lat_max', None),
            'lon_min': zone_info.get('lon_min', None),
            'lon_max': zone_info.get('lon_max', None),
            'description': info['description']
        })

df_correspondance = pd.DataFrame(correspondances)

print(" TABLE DE CORRESPONDANCE CRÉÉE")
print("=" * 70)
print(f"Nombre d'entrées: {len(df_correspondance)}")
print(f"Zones uniques: {df_correspondance['zone_cotiere'].nunique()}")
print(f"Régions couvertes: {df_correspondance['region'].nunique()}")

print("\n Aperçu:")
display(df_correspondance)

 TABLE DE CORRESPONDANCE CRÉÉE
Nombre d'entrées: 43
Zones uniques: 12
Régions couvertes: 13

 Aperçu:


,zone_cotiere,region,cote,lat_min,lat_max,lon_min,lon_max,description
0,CAP D'AMBRE A TOAMASINA,DIANA,EST,-18.5,-11.5,49.0,50.5,Côte nord-est de Madagascar
1,CAP D'AMBRE A TOAMASINA,SAVA,EST,-18.5,-11.5,49.0,50.5,Côte nord-est de Madagascar
2,CAP D'AMBRE A TOAMASINA,ANALANJIROFO,EST,-18.5,-11.5,49.0,50.5,Côte nord-est de Madagascar
3,CAP D'AMBRE A TOAMASINA,ATSINANANA,EST,-18.5,-11.5,49.0,50.5,Côte nord-est de Madagascar
4,CAP D'AMBRE A MAHANORO,DIANA,EST,-20.0,-11.5,48.5,50.5,Côte nord-est étendue jusqu'à Mahanoro
5,CAP D'AMBRE A MAHANORO,SAVA,EST,-20.0,-11.5,48.5,50.5,Côte nord-est étendue jusqu'à Mahanoro
6,CAP D'AMBRE A MAHANORO,ANALANJIROFO,EST,-20.0,-11.5,48.5,50.5,Côte nord-est étendue jusqu'à Mahanoro
7,CAP D'AMBRE A MAHANORO,ATSINANANA,EST,-20.0,-11.5,48.5,50.5,Côte nord-est étendue jusqu'à Mahanoro
8,CAP D'AMBRE A ANTALAHA,DIANA,EST,-15.0,-11.5,49.0,50.5,Extrême nord-est
9,CAP D'AMBRE A ANTALAHA,SAVA,EST,-15.0,-11.5,49.0,50.5,Extrême nord-est


### STATISTIQUES DU MAPPING

In [7]:
print(" STATISTIQUES DU MAPPING")
print("=" * 70)

# Régions par zone
print("\n Nombre de régions par zone:")
regions_par_zone = df_correspondance.groupby('zone_cotiere')['region'].count()
print(regions_par_zone.sort_values(ascending=False))

# Zones par région
print("\n Nombre de zones par région:")
zones_par_region = df_correspondance.groupby('region')['zone_cotiere'].count()
print(zones_par_region.sort_values(ascending=False))

 STATISTIQUES DU MAPPING

 Nombre de régions par zone:
zone_cotiere
CAP EST A TAOLAGNARO             5
TOAMASINA AU CAP SAINTE MARIE    5
MAHANORO AU CAP SAINTE MARIE     5
CAP D'AMBRE A BESALAMPY          4
CAP D'AMBRE A MAHANORO           4
TOAMASINA A TAOLAGNARO           4
CAP D'AMBRE A TOAMASINA          4
BESALAMPY A MOROMBE              3
MOROMBE A TAOLAGNARO             3
CAP D'AMBRE A ANTALAHA           2
CAP D'AMBRE A CAP EST            2
MOROMBE AU CAP SAINTE MARIE      2
Name: region, dtype: int64

 Nombre de zones par région:
region
ATSINANANA             6
DIANA                  5
ANOSY                  5
ATSIMO ATSINANANA      4
ANDROY                 4
VATOVAVY FITOVINANY    4
SAVA                   4
ATSIMO ANDREFANA       3
ANALANJIROFO           3
MELAKY                 2
BOENY                  1
MENABE                 1
SOFIA                  1
Name: zone_cotiere, dtype: int64


## 6. Validation avec les Incidents Maritimes

Vérifions que notre mapping couvre bien les régions où se produisent les incidents maritimes.

In [8]:
# Charger les données des incidents maritimes
df_incidents = pd.read_csv(os.path.join(CSV_PATH, 'incidents_maritimes_complets_2017_2022.csv'))

# Filtrer les vrais incidents
df_vrais_incidents = df_incidents[~df_incidents['description'].str.contains('Aucun incident', na=True)]

print(" INCIDENTS MARITIMES")
print("=" * 70)
print(f"Total incidents: {len(df_vrais_incidents)}")

# Régions des incidents
regions_incidents = df_vrais_incidents['Region'].dropna().unique()
print(f"\n Régions des incidents ({len(regions_incidents)} uniques):")
for region in sorted(regions_incidents):
    if pd.notna(region) and str(region).strip():
        print(f"   • {region}")

 INCIDENTS MARITIMES
Total incidents: 269

 Régions des incidents (9 uniques):
   • ANALANJIROFO
   • ATSIMO ANDREFANA
   • ATSIMO ATSINANANA
   • ATSINANANA
   • BOENY
   • EST
   • MELAKY
   • NORD
   • SOFIA


In [9]:

# Régions dans notre mapping
regions_mapping = set(df_correspondance['region'].str.upper())

# Régions des incidents (nettoyées)
regions_incidents_clean = set()
for region in regions_incidents:
    if pd.notna(region) and str(region).strip():
        regions_incidents_clean.add(str(region).upper().strip())

# Comparaison
print("🔍 VALIDATION DE LA COUVERTURE")
print("=" * 70)

regions_couvertes = regions_mapping.intersection(regions_incidents_clean)
regions_non_couvertes = regions_incidents_clean - regions_mapping

print(f"\n Régions couvertes par le mapping: {len(regions_couvertes)}")
for r in sorted(regions_couvertes):
    print(f"   • {r}")

if regions_non_couvertes:
    print(f"\n Régions NON couvertes (intérieures?): {len(regions_non_couvertes)}")
    for r in sorted(regions_non_couvertes):
        print(f"   • {r}")
else:
    print("\n Toutes les régions des incidents sont couvertes!")

🔍 VALIDATION DE LA COUVERTURE

 Régions couvertes par le mapping: 7
   • ANALANJIROFO
   • ATSIMO ANDREFANA
   • ATSIMO ATSINANANA
   • ATSINANANA
   • BOENY
   • MELAKY
   • SOFIA

 Régions NON couvertes (intérieures?): 2
   • EST
   • NORD


##  7. Fonction d'Attribution de Zone

Cette fonction permet d'attribuer une **zone côtière** à un incident en fonction de sa région ou de ses coordonnées.

In [ ]:
def attribuer_zone_cotiere(region: str = None, latitude: float = None, 
                           longitude: float = None, df_mapping: pd.DataFrame = None) -> list:
    """
    Attribue une ou plusieurs zones côtières à un incident.
    
    Args:
        region: Nom de la région administrative
        latitude: Latitude de l'incident
        longitude: Longitude de l'incident
        df_mapping: DataFrame de correspondance zones-régions
        
    Returns:
        Liste des zones côtières correspondantes
    """
    zones = []
    
    if df_mapping is None:
        df_mapping = df_correspondance
    
    # Méthode 1: Par région
    if region and pd.notna(region):
        region_upper = str(region).upper().strip()
        zones_region = df_mapping[df_mapping['region'].str.upper() == region_upper]['zone_cotiere'].tolist()
        zones.extend(zones_region)
    
    # Méthode 2: Par coordonnées (si disponibles)
    if latitude and longitude and pd.notna(latitude) and pd.notna(longitude):
        for zone, info in ZONES_COTIERES.items():
            if (info['lat_min'] <= latitude <= info['lat_max'] and
                info['lon_min'] <= longitude <= info['lon_max']):
                if zone not in zones:
                    zones.append(zone)
    
    return list(set(zones)) if zones else ['ZONE_INCONNUE']

# Test de la fonction
print("🧪 TEST DE LA FONCTION D'ATTRIBUTION")
print("=" * 70)

tests = [
    {"region": "DIANA", "lat": None, "lon": None},
    {"region": "ATSINANANA", "lat": -18.5, "lon": 49.4},
    {"region": "ANOSY", "lat": -25.0, "lon": 47.0},
    {"region": "BOENY", "lat": -15.7, "lon": 46.3}
]

for test in tests:
    zones = attribuer_zone_cotiere(test['region'], test['lat'], test['lon'])
    print(f"\n Région: {test['region']}, Coords: ({test['lat']}, {test['lon']})")
    print(f"    Zones: {zones}")

🧪 TEST DE LA FONCTION D'ATTRIBUTION

 Région: DIANA, Coords: (None, None)
    Zones: ["CAP D'AMBRE A TOAMASINA", "CAP D'AMBRE A CAP EST", "CAP D'AMBRE A ANTALAHA", "CAP D'AMBRE A MAHANORO", "CAP D'AMBRE A BESALAMPY"]

 Région: ATSINANANA, Coords: (-18.5, 49.4)
    Zones: ["CAP D'AMBRE A TOAMASINA", "CAP D'AMBRE A MAHANORO", 'TOAMASINA AU CAP SAINTE MARIE', 'MAHANORO AU CAP SAINTE MARIE', 'TOAMASINA A TAOLAGNARO', 'CAP EST A TAOLAGNARO']

 Région: ANOSY, Coords: (-25.0, 47.0)
    Zones: ['TOAMASINA AU CAP SAINTE MARIE', 'MAHANORO AU CAP SAINTE MARIE', 'TOAMASINA A TAOLAGNARO', 'MOROMBE A TAOLAGNARO', 'CAP EST A TAOLAGNARO']

 Région: BOENY, Coords: (-15.7, 46.3)
    Zones: ["CAP D'AMBRE A BESALAMPY"]


##  8. Sauvegarde des Tables de Correspondance

In [ ]:
# 1. Table de correspondance principale (CSV)
output_csv = os.path.join(OUTPUT_PATH, 'correspondance_zones_regions.csv')
df_correspondance.to_csv(output_csv, index=False, encoding='utf-8')
print(f" Sauvegardé: {output_csv}")

# 2. Mapping complet en JSON (pour utilisation programmatique)
output_json = os.path.join(OUTPUT_PATH, 'mapping_zones_regions.json')
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump({
        'zones_cotieres': ZONES_COTIERES,
        'mapping': MAPPING_ZONES_REGIONS,
        'points_repere': POINTS_REPERE
    }, f, indent=2, ensure_ascii=False)
print(f" Sauvegardé: {output_json}")

# 3. Tableau pivot zones-régions
pivot = df_correspondance.pivot_table(
    index='zone_cotiere', 
    columns='region', 
    aggfunc=lambda x: 1,
    fill_value=0
)
output_pivot = os.path.join(OUTPUT_PATH, 'matrice_zones_regions.csv')
pivot.to_csv(output_pivot)
print(f" Sauvegardé: {output_pivot}")

print("\n TOUS LES FICHIERS SAUVEGARDÉS")

 Sauvegardé: /home/henintsoa/CFIM/final/data/correspondance_zones_regions.csv
 Sauvegardé: /home/henintsoa/CFIM/final/data/mapping_zones_regions.json


 Sauvegardé: /home/henintsoa/CFIM/final/data/matrice_zones_regions.csv

 TOUS LES FICHIERS SAUVEGARDÉS


##  9. Visualisation du Mapping (Matrice)

In [ ]:
print(" MATRICE DE CORRESPONDANCE ZONES ↔ RÉGIONS")
print("=" * 70)
print("(1 = correspondance, 0 = pas de correspondance)\n")

# Créer une matrice plus lisible
regions_liste = sorted(df_correspondance['region'].unique())
zones_liste = sorted(df_correspondance['zone_cotiere'].unique())

matrice = []
for zone in zones_liste:
    row = {'Zone': zone}
    regions_zone = df_correspondance[df_correspondance['zone_cotiere'] == zone]['region'].tolist()
    for region in regions_liste:
        row[region] = '✓' if region in regions_zone else ''
    matrice.append(row)

df_matrice = pd.DataFrame(matrice)
df_matrice = df_matrice.set_index('Zone')

display(df_matrice)

 MATRICE DE CORRESPONDANCE ZONES ↔ RÉGIONS
(1 = correspondance, 0 = pas de correspondance)



,ANALANJIROFO,ANDROY,ANOSY,ATSIMO ANDREFANA,ATSIMO ATSINANANA,ATSINANANA,BOENY,DIANA,MELAKY,MENABE,SAVA,SOFIA,VATOVAVY FITOVINANY
Zone,,,,,,,,,,,,,
BESALAMPY A MOROMBE,,,,✓,,,,,✓,✓,,,
CAP D'AMBRE A ANTALAHA,,,,,,,,✓,,,✓,,
CAP D'AMBRE A BESALAMPY,,,,,,,✓,✓,✓,,,✓,
CAP D'AMBRE A CAP EST,,,,,,,,✓,,,✓,,
CAP D'AMBRE A MAHANORO,✓,,,,,✓,,✓,,,✓,,
CAP D'AMBRE A TOAMASINA,✓,,,,,✓,,✓,,,✓,,
CAP EST A TAOLAGNARO,✓,,✓,,✓,✓,,,,,,,✓
MAHANORO AU CAP SAINTE MARIE,,✓,✓,,✓,✓,,,,,,,✓
MOROMBE A TAOLAGNARO,,✓,✓,✓,,,,,,,,,




         CORRESPONDANCE SPATIALE ZONES-RÉGIONS - TERMINÉE           
                                                                    
  Résultats:                                                         
    • 12 zones côtières définies                                     
    • 15 régions côtières mappées                                    
    • 42 correspondances zone ↔ région créées                        
                                                                     
  Fichiers créés:                                                    
    • correspondance_zones_regions.csv                               
    • mapping_zones_regions.json                                     
    • matrice_zones_regions.csv                                      
                                                                     
  Couverture:                                                        
    • Toutes les régions côtières avec incidents sont mappées        
    • Les régions intérieures sont exclues (pas de lien maritime)    
                                                                     
                                                                     